In [0]:
# Investigate Silver data-quality conditions for the Toronto snapshot.
#
# This notebook is for operational investigation. It does not replace
# the production Silver cleaning notebook.

from pyspark.sql import functions as F

CATALOG = "trippilot_dev"
SILVER_SCHEMA = "silver"

SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.airbnb_listings_silver"
REJECTED_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.airbnb_listings_rejected"

silver_df = spark.table(SILVER_TABLE)
rejected_df = spark.table(REJECTED_TABLE)

quality_summary = silver_df.agg(
    F.count("*").alias("silver_rows"),
    F.count(F.when(F.col("has_valid_price") == False, 1)).alias("missing_price"),
    F.count(F.when(F.col("has_valid_coordinates") == False, 1)).alias("missing_coordinates"),
    F.count(
        F.when(
            (F.col("has_valid_price") == False)
            & (F.col("has_valid_coordinates") == False),
            1,
        )
    ).alias("missing_price_and_coordinates"),
)

display(quality_summary)

display(
    rejected_df.select(
        "listing_id",
        "listing_name",
        "neighbourhood",
        "room_type",
        "price_raw",
        "latitude_raw",
        "longitude_raw",
        "source",
        "city",
        "snapshot_date",
    ).limit(50)
)